In [ ]:
function [rgb] = func_hyperImshow( hsi, RGBbands )
%% Hyperspectral Image color display
% Author: Zephyr Hou, Di Song, Yuchen Wang
% Time: 2019-12-02
% Function Usage
% Input:
%    hsi—the 3D hyperspectral dataset with the size of rows x cols x bands
%    RGBbands— the RGB bands to be displayed, with the format [R G B]
% Output:
%    rgb- the finual result with the RGB bands with the size of (rows x cols x 3)
%% Main Function

hsi=double(hsi);
[rows, cols, bands] = size(hsi);

minVal =min(hsi(:));
maxVal=max(hsi(:));
normalizedData=hsi-minVal;

if(maxVal==minVal)
    normalizedData=zeros(size(hsi));
else
    normalizedData=normalizedData./(maxVal-minVal);
end

hsi=normalizedData;

[rows, cols, bands] = size(hsi);

if (nargin == 1)
    RGBbands = [bands round(bands/2) 1];
end

if (bands ==1)
    red = hsi(:,:);
    green = hsi(:,:);
    blue = hsi(:,:);
else
    red = hsi(:,:,RGBbands(1));
    green = hsi(:,:,RGBbands(2));
    blue = hsi(:,:,RGBbands(3));
end

rgb = zeros(size(hsi, 1), size(hsi, 2), 3);
rgb(:,:,1) = adapthisteq(red);
rgb(:,:,2) = adapthisteq(green);
rgb(:,:,3) = adapthisteq(blue);
imshow(rgb); axis image;
end

SyntaxError: invalid syntax (3536994545.py, line 16)

In [ ]:
function [hdrstruct] = hdrread(str)


%     str='1_ref.hdr';
fid=fopen(str,'r');
info = fread(fid, 'char=>char');
info=info';
fclose(fid);

%% sample大小
start = strfind(info,'samples = ');
len = length('samples = ');
stop = strfind(info,'bands');
samples = [];
for i = start+len : stop-1
    samples = [samples, info(i)];
end
samples = str2num(samples);

%% 获取size大小
start = strfind(info,'lines = ');
len = length('lines = ');
stop = strfind(info,'samples');
lines = [];
for i = start+len : stop-1
    lines = [lines, info(i)];
end
lines = str2num(lines);
%% 获取lines
start = strfind(info,'bands = ');
len = length('bands = ');
stop = strfind(info,'sample binning');
bands = [];
for i = start+len : stop-1
    bands = [bands, info(i)];
end
bands = str2num(bands);
%% 获取datatype
start = strfind(info,'data type = ');
len = length('data type = ');
stop = strfind(info,'lines');
datatype = [];
for i = start+len : stop-1
    datatype = [datatype, info(i)];
end
datatype = str2num(datatype);

%% 获取precision， data__type
precision = [];
switch datatype
    case 1
        precision = 'uint8 => uint8';
    case 2
        precision = 'int16 => int16';
    case 12
        precision = 'uint16 => uint16';
    case 3
        precision = 'int32 => int32';
    case 13
        precision = 'uint32 => uint32';
    case 4
        precision = 'float32 => float32';
    case 5
        precision = 'double => double';
    otherwise
        precision = 'invalid type';
end
%% 获取interleave
start = strfind(info,'interleave = ');
len = length('interleave = ');
stop = strfind(info,'byte order');
interleave = [];
for i = start+len : stop-1
    interleave = [interleave, info(i)];
end
interleave = strtrim(interleave);
%% wavelength
% start = strfind(info,'wavelength = {');
% len = length('wavelength = {');
% stop = length(info);
% wavelength = [];
% for i = start+len : stop-1
%     wavelength = [wavelength, info(i)];
% end
% s=textscan(wavelength,'%s','delimiter',',')
% wavelength=(zeros(bands,1));
% for i=1:bands
%     wavelength(i,1)=str2num(s{1,1}{i,1});
% end

hdrstruct.samples=samples;
hdrstruct.lines=lines;
hdrstruct.bands=bands;
hdrstruct.datatypes=datatype;
hdrstruct.precision=precision;
hdrstruct.interleave=interleave;
% hdrstruct.wavelength=wavelength;
end

SyntaxError: unterminated string literal (detected at line 7) (1751918631.py, line 7)

In [ ]:
clc;
clear;
R=[];
C=[];

% Process single file: 16.bil and 16.bil.hdr
filePath = '16.bil';
hdr_name = '16.bil.hdr';

N=[];
hdr_info = hdrread(hdr_name);
rows = hdr_info.lines;            % 行数
cols = hdr_info.samples;          % 列数
bands = hdr_info.bands;           % 波段数
interleave = hdr_info.interleave; % 数据排列方式（应该是 'bip'）

spec = multibandread(filePath, [rows cols bands], 'uint16', 0, 'bil', 'ieee-le');
spec= spec./ 10000;
[l,r,d]=size(spec);

% 3 bands precess
scope1=(spec(:,:,90)-spec(:,:,69))/15*100;
scope2=(spec(:,:,109)-spec(:,:,90))/13*100;
scope3=(spec(:,:,109)-spec(:,:,69))/28;

for i=1:l
   for j=1:r
       N(i,j) = scope1(i,j)-scope2(i,j);
   end
end

image = func_hyperImshow(N, [1,2,3]);
image2 = func_hyperImshow(spec, [128,85,42]);
imwrite(image,'16_enhance.jpg','jpg');
imwrite(image2,'16_original.jpg','jpg');